In [ ]:
#pip install pyspark

In [1]:
import os
import sys
from pyspark.sql import SparkSession

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
spark = SparkSession.builder.getOrCreate()

In [2]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, BooleanType,\
TimestampType, DateType
from pyspark.sql.functions import sum, col, count, when, lead, collect_list, length, year, \
countDistinct, round, first, lit
from datetime import datetime

In [4]:
spark = SparkSession.builder.master('local[*]').appName('PysparkDataPipeline').getOrCreate()

#### Creating empty dataframe - 

In [5]:
schemac = StructType([StructField('col1',IntegerType(),True),
                      StructField('col2',StringType(),False)])

In [6]:
empty_df = spark.createDataFrame([],schema = schemac)

In [7]:
empty_df.show()

+----+----+
|col1|col2|
+----+----+
+----+----+



#### Append data into it 

In [8]:
empty_df = spark.createDataFrame([[1,'a'],
                                  [2,'b'],
                                  [3,'c'],
                                  [None,'d']],schema = schemac)
empty_df.show()

+----+----+
|col1|col2|
+----+----+
|   1|   a|
|   2|   b|
|   3|   c|
|NULL|   d|
+----+----+



#### Filling Null values - 

In [9]:
empty_df = empty_df.fillna(1,subset = ['col1'])
empty_df.show()

+----+----+
|col1|col2|
+----+----+
|   1|   a|
|   2|   b|
|   3|   c|
|   1|   d|
+----+----+



#### Replacing 1s with 2s 

In [10]:
empty_df = empty_df.replace(1,2, subset = ['col1'])
empty_df.show()

+----+----+
|col1|col2|
+----+----+
|   2|   a|
|   2|   b|
|   3|   c|
|   2|   d|
+----+----+



#### Leetcode 1: Replace Employee ID with Unique Identifier

In [11]:
employees_schema = StructType([StructField('id', IntegerType(), False),\
                               StructField('name', StringType(), True)])
employees_df = spark.createDataFrame([[1, 'Alice'],
                                      [7, 'Bob'],
                                      [11, 'Meir'],
                                      [90, 'Winston'],
                                      [3, 'Jonathan']], schema=employees_schema)
print(employees_df)
employee_uni_schema = StructType([StructField('id', IntegerType(), False),\
                                  StructField('unique_id', IntegerType(), True)])

employee_uni_df = spark.createDataFrame([[3, 1],
                                         [11, 2],
                                         [90, 3]], schema=employee_uni_schema)
print(employee_uni_df)

DataFrame[id: int, name: string]
DataFrame[id: int, unique_id: int]


In [12]:
employees_df.show()

+---+--------+
| id|    name|
+---+--------+
|  1|   Alice|
|  7|     Bob|
| 11|    Meir|
| 90| Winston|
|  3|Jonathan|
+---+--------+



In [13]:
employee_uni_df.show()

+---+---------+
| id|unique_id|
+---+---------+
|  3|        1|
| 11|        2|
| 90|        3|
+---+---------+



In [14]:
employees_df.join(employee_uni_df,how = 'left', on = ['id']).select('unique_id','name').show()

+---------+--------+
|unique_id|    name|
+---------+--------+
|     NULL|   Alice|
|     NULL|     Bob|
|        2|    Meir|
|        3| Winston|
|        1|Jonathan|
+---------+--------+



#### Confirmation Rate

In [16]:
signups_schema = StructType([StructField('user_id', IntegerType(), False),
                             StructField('time_stamp', TimestampType(), True)])
confirmations_schema = StructType([StructField('user_id', IntegerType(), True),
                                   StructField('time_stamp', TimestampType(), True),
                                   StructField('action', StringType(), True)])

In [17]:
sinups_df = spark.createDataFrame([[3, datetime.strptime('2020-03-21 10:16:13', '%Y-%m-%d %H:%M:%S')],
                                   [7, datetime.strptime('2020-01-04 13:57:59', '%Y-%m-%d %H:%M:%S')],
                                   [2, datetime.strptime('2020-07-29 23:09:44', '%Y-%m-%d %H:%M:%S')],
                                   [6, datetime.strptime('2020-12-09 10:39:37', '%Y-%m-%d %H:%M:%S')]], 
                                  schema=signups_schema)
confirmations_df = spark.createDataFrame([[3, datetime.strptime('2021-01-06 03:30:46', '%Y-%m-%d %H:%M:%S'), 'timeout'],
                                          [3, datetime.strptime('2021-07-14 14:00:00', '%Y-%m-%d %H:%M:%S'), 'timeout'],
                                          [7, datetime.strptime('2021-06-12 11:57:29', '%Y-%m-%d %H:%M:%S'), 'confirmed'],
                                          [7, datetime.strptime('2021-06-13 12:58:28', '%Y-%m-%d %H:%M:%S'), 'confirmed'],
                                          [7, datetime.strptime('2021-06-14 13:59:27', '%Y-%m-%d %H:%M:%S'), 'confirmed'],
                                          [2, datetime.strptime('2021-01-22 00:00:00', '%Y-%m-%d %H:%M:%S'), 'confirmed'],
                                          [2, datetime.strptime('2021-02-28 23:59:59', '%Y-%m-%d %H:%M:%S'), 'timeout']], 
                                         schema=confirmations_schema)

In [18]:
result_df = sinups_df.join(confirmations_df, how = 'left', on = ['user_id'])

In [19]:
result_df.groupBy('user_id').agg((sum(when(result_df.action == 'confirmed',1).otherwise(0.00))/
                                count('*')).alias('Confirmation_Rate')).show()

+-------+-----------------+
|user_id|Confirmation_Rate|
+-------+-----------------+
|      3|              0.0|
|      7|              1.0|
|      2|              0.5|
|      6|              0.0|
+-------+-----------------+



#### Find users with Valid Emails -  

In [20]:
schema = StructType([StructField('user_id', IntegerType(), False),
                     StructField('name', StringType(), True),
                     StructField('mail', StringType(), True)])

df = spark.createDataFrame([[1, 'Winston', 'winston@leetcode.com'],
                            [2, 'Jonathan', 'jonathanisgreat'],
                            [3, 'Annabelle', 'bella-@leetcode.com'],
                            [4, 'Sally', 'sally.come@leetcode.com'],
                            [5, 'Marwan', 'quarz#2020@leetcode.com'], 
                            [6, 'David', 'david69@gmail.com'],
                            [7, 'Shapiro', '.shapo@leetcode.com']], schema=schema)

In [21]:
df.show()

+-------+---------+--------------------+
|user_id|     name|                mail|
+-------+---------+--------------------+
|      1|  Winston|winston@leetcode.com|
|      2| Jonathan|     jonathanisgreat|
|      3|Annabelle| bella-@leetcode.com|
|      4|    Sally|sally.come@leetco...|
|      5|   Marwan|quarz#2020@leetco...|
|      6|    David|   david69@gmail.com|
|      7|  Shapiro| .shapo@leetcode.com|
+-------+---------+--------------------+



In [22]:
df.filter(col('mail').rlike('^[a-zA-Z][a-zA-Z0-9_.-]*@leetcode.com')).show()

+-------+---------+--------------------+
|user_id|     name|                mail|
+-------+---------+--------------------+
|      1|  Winston|winston@leetcode.com|
|      3|Annabelle| bella-@leetcode.com|
|      4|    Sally|sally.come@leetco...|
+-------+---------+--------------------+



#### Consecutive Numbers appearing at least 3 times

In [23]:
schema = StructType([StructField('id',IntegerType(),False),
                     StructField('num',StringType(),True)])
df = spark.createDataFrame([[1, '1'],
                            [2, '1'],
                            [3, '1'],
                            [4, '2'],
                            [5, '3'],
                            [6, '3'],
                            [7, '3']], schema=schema)

In [24]:
df.printSchema()

root
 |-- id: integer (nullable = false)
 |-- num: string (nullable = true)



In [25]:
df1 = df.createOrReplaceTempView('df')

In [26]:
df2 = spark.sql(''' with tmp as (
          SELECT NUM,lead(num, 1) over (order by id) num1,lead(num, 2) over (order by id) num2 
          from df)
          SELECT DISTINCT NUM FROM TMP WHERE NUM = NUM1 AND NUM = NUM2''').show()

+---+
|NUM|
+---+
|  1|
|  3|
+---+



In [27]:
df.withColumn('num1',lead('num',1).over(Window.orderBy('id')))\
.withColumn('num2',lead('num',2).over(Window.orderBy('id')))\
.filter((col('num') == col('num1')) & (col('num') == col('num2')))\
.select('id').distinct().show()

+---+
| id|
+---+
|  1|
|  5|
+---+



#### Students & Classes

In [29]:
schema = StructType([StructField('class', IntegerType(), True),
                     StructField('student_id', IntegerType(), True),
                     StructField('term', IntegerType(), True),
                     StructField('subject', StringType(), True),
                     StructField('marks', IntegerType(), True)])
df = spark.createDataFrame([[2, 1, 1, 'maths', 10],
                            [2, 1, 2, 'maths', 12],
                             [2, 1, 1, 'english', 14],
                             [2, 1, 2, 'english', 12],
                             [3, 2, 1, 'maths', 10],
                             [3, 2, 2, 'maths', 12],
                             [3, 2, 1, 'english', 16],
                             [3, 2, 2, 'english', 14]], schema=schema)


In [30]:
df.show()

+-----+----------+----+-------+-----+
|class|student_id|term|subject|marks|
+-----+----------+----+-------+-----+
|    2|         1|   1|  maths|   10|
|    2|         1|   2|  maths|   12|
|    2|         1|   1|english|   14|
|    2|         1|   2|english|   12|
|    3|         2|   1|  maths|   10|
|    3|         2|   2|  maths|   12|
|    3|         2|   1|english|   16|
|    3|         2|   2|english|   14|
+-----+----------+----+-------+-----+



#### Get class 2 students in following format --> class student_id subject term1 term2

In [31]:
dfn = df.orderBy('class','student_id','subject')\
.groupBy('student_id','subject','class')\
.agg(collect_list('marks')[0].alias('Term1'),collect_list('marks')[1].alias('Term2'))

In [32]:
dfn.show()

+----------+-------+-----+-----+-----+
|student_id|subject|class|Term1|Term2|
+----------+-------+-----+-----+-----+
|         1|english|    2|   14|   12|
|         1|  maths|    2|   10|   12|
|         2|english|    3|   16|   14|
|         2|  maths|    3|   10|   12|
+----------+-------+-----+-----+-----+



In [34]:
dfn.filter(col('subject') == 'english').show()

+----------+-------+-----+-----+-----+
|student_id|subject|class|Term1|Term2|
+----------+-------+-----+-----+-----+
|         1|english|    2|   14|   12|
|         2|english|    3|   16|   14|
+----------+-------+-----+-----+-----+



#### Get subject-wise aggregated score with 25% weightage to term1 and 75% weightage to term2

In [35]:
df = spark.createDataFrame([[2, 1, 1, 'maths', 10],
 [2, 1, 2, 'maths', 12],
 [2, 1, 1, 'english', 14],
 [2, 1, 2, 'english', 12],
 [3, 2, 1, 'maths', 10],
 [3, 2, 2, 'maths', 12],
 [3, 2, 1, 'english', 16],
 [3, 2, 2, 'english', 14]], schema=schema)

In [36]:
df.show()

+-----+----------+----+-------+-----+
|class|student_id|term|subject|marks|
+-----+----------+----+-------+-----+
|    2|         1|   1|  maths|   10|
|    2|         1|   2|  maths|   12|
|    2|         1|   1|english|   14|
|    2|         1|   2|english|   12|
|    3|         2|   1|  maths|   10|
|    3|         2|   2|  maths|   12|
|    3|         2|   1|english|   16|
|    3|         2|   2|english|   14|
+-----+----------+----+-------+-----+



In [37]:
maths_agg = df.filter(col('subject') == 'maths')\
.orderBy('student_id','term')\
.groupBy('student_id')\
.agg((collect_list('marks')[0]*0.25 + collect_list('marks')[1]*0.75).alias('maths_agg'))

In [38]:
english_agg = df.filter(col('subject') == 'english')\
.orderBy('student_id','term')\
.groupBy('student_id')\
.agg((collect_list('marks')[0]*0.25 + collect_list('marks')[1]*0.75).alias('english_agg'))

In [39]:
maths_agg.join(english_agg, how = 'inner', on = 'student_id').show()

+----------+---------+-----------+
|student_id|maths_agg|english_agg|
+----------+---------+-----------+
|         1|     11.5|       12.5|
|         2|     11.5|       14.5|
+----------+---------+-----------+



#### Exchange Seats -  

In [40]:
import pandas as pd
import io
data = '''
id,student
1,Abbot
2,Doris
3,Emerson
4,Green
5,Jeames
'''
df = spark.createDataFrame(pd.read_csv(io.StringIO(data), header=0))

In [41]:
df.show()

+---+-------+
| id|student|
+---+-------+
|  1|  Abbot|
|  2|  Doris|
|  3|Emerson|
|  4|  Green|
|  5| Jeames|
+---+-------+



In [42]:
print(df.select('id').collect())

[Row(id=1), Row(id=2), Row(id=3), Row(id=4), Row(id=5)]


In [43]:
max_id = max(df.select('id').collect())[0]
max_id

5

In [44]:
df.select(when(df['id']%2==1,
              lead('id',1,max_id).over(Window.orderBy('id')))
         .otherwise(col('id')-1).alias('id'),'Student').orderBy('id').show()

+---+-------+
| id|Student|
+---+-------+
|  1|  Doris|
|  2|  Abbot|
|  3|  Green|
|  4|Emerson|
|  5| Jeames|
+---+-------+



#### Tree Node -

In [45]:
schema = StructType([StructField('id', IntegerType(), False),
 StructField('p_id', IntegerType(), True)])
df = spark.createDataFrame(
 [[1, None],
 [2, 1],
 [3, 1],
 [4, 2],
 [5, 2]], schema=schema)

In [46]:
df.show()

+---+----+
| id|p_id|
+---+----+
|  1|NULL|
|  2|   1|
|  3|   1|
|  4|   2|
|  5|   2|
+---+----+



In [47]:
distinct_parent_ids = df.select('p_id').distinct().rdd.flatMap(lambda x: x).collect()

In [48]:
df.select('id',
 when(col('p_id').isNull(), 'Root')
 .when(col('p_id').isNotNull() & col('id').isin(distinct_parent_ids), 'Inner')
 .otherwise('Leaf').alias('type')).show()

+---+-----+
| id| type|
+---+-----+
|  1| Root|
|  2|Inner|
|  3| Leaf|
|  4| Leaf|
|  5| Leaf|
+---+-----+

